In [ ]:
#converting sdf to pdbqt
import os
from openbabel import pybel

# Set your folder path here
folder_path = '/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/final MT/before gnina 3D'
for sdf_file in os.listdir(folder_path):
    if not sdf_file.lower().endswith('.sdf'):
        continue
    sdf_path = os.path.join(folder_path, sdf_file)
    base_name = os.path.splitext(sdf_file)[0]

    for idx, mol in enumerate(pybel.readfile("sdf", sdf_path), start=1):
        mol.addh()                            # ← add explicit H’s
        mol.calccharges(model="gasteiger")    # ← assign Gasteiger charges

        pdbqt_file = f"{base_name}_{idx}.pdbqt"
        pdbqt_path = os.path.join(folder_path, pdbqt_file)
        mol.write("pdbqt", pdbqt_path, overwrite=True)
print("All conversions complete!")


#for removing _1 from pdbqt name uisng cli
#for f in *_1.*; do
#  mv -- "$f" "${f/_1/}"
#done


All conversions complete!


In [1]:
#Optimized Script for Extracting Top CNNaffinity Pose and Saving CSV along with top pose pdbqt, from pdbqt in batch
import os
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/final MT/gnina/ABL1')
import glob
import os
import re
import csv

input_dir = "3_docked_logs"
output_csv = "3_top_pose_scores.csv"
csv_rows = []
header = ["Ligand", "minimizedAffinity", "CNNaffinity", "CNNscore"]

for infile in glob.glob(os.path.join(input_dir, "docked_*.pdbqt")):
    ligand_name = os.path.splitext(os.path.basename(infile))[0]
    with open(infile) as f:
        lines = f.readlines()

    poses = []
    current_pose = []
    current_cnn = None
    current_min_aff = None
    current_cnnscore = None

    for line in lines:
        if line.startswith("MODEL"):
            current_pose = [line]
            current_cnn = None
            current_min_aff = None
            current_cnnscore = None
        elif line.startswith("ENDMDL"):
            current_pose.append(line)
            if current_cnn is not None:
                poses.append((current_cnn, current_min_aff, current_cnnscore, list(current_pose)))
            current_pose = []
        else:
            current_pose.append(line)
            if line.startswith("REMARK minimizedAffinity"):
                m = re.search(r"REMARK minimizedAffinity\s+([-\d\.]+)", line)
                if m:
                    current_min_aff = float(m.group(1))
            if line.startswith("REMARK CNNaffinity"):
                m = re.search(r"REMARK CNNaffinity\s+([-\d\.]+)", line)
                if m:
                    current_cnn = float(m.group(1))
            if line.startswith("REMARK CNNscore"):
                m = re.search(r"REMARK CNNscore\s+([-\d\.]+)", line)
                if m:
                    current_cnnscore = float(m.group(1))

    # Select pose with highest CNNaffinity (not lowest)
    if poses:
        best = max(poses, key=lambda x: x[0])  # highest CNNaffinity
        best_cnn, best_min_aff, best_cnnscore, best_pose = best

        out_pdbqt = os.path.join(input_dir, ligand_name + "_top_cnn.pdbqt")
        with open(out_pdbqt, "w") as out:
            out.writelines(best_pose)

        csv_rows.append({
            "Ligand": ligand_name,
            "minimizedAffinity": best_min_aff,
            "CNNaffinity": best_cnn,
            "CNNscore": best_cnnscore
        })
    else:
        print(f"No poses found in {infile}")

with open(output_csv, "w", newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=header)
    writer.writeheader()
    for row in csv_rows:
        writer.writerow(row)

print(f"Top pose scores saved to {output_csv}")


Top pose scores saved to 3_top_pose_scores.csv


In [ ]:
import os
import pandas as pd
from functools import reduce

# ─── User configuration ───────────────────────────────────────────────────────
os.chdir('/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT/final MT/gnina/')

# Input files
files = {
    "DYRK1A": os.path.join("DYRK1A", "2_top_pose_scores.csv"),
    "ABL1":   os.path.join("ABL1",   "2_top_pose_scores.csv"),
    "TTBK1":  os.path.join("TTBK1",  "2_top_pose_scores.csv"),
}

# Base columns
key_col       = "Ligand"
score_cols    = ["minimizedAffinity", "CNNaffinity", "CNNscore"]
output_csv    = "3_combined_all_targets.csv"
# ─────────────────────────────────────────────────────────────────────────────

# Load, rename, and collect DataFrames
dfs = []
for target, file in files.items():
    df = pd.read_csv(file)
    df = df[[key_col] + score_cols].copy()
    df.rename(columns={col: f"{col}_{target}" for col in score_cols}, inplace=True)
    dfs.append(df)

# Merge all DataFrames on Ligand
merged_df = reduce(lambda left, right: pd.merge(left, right, on=key_col, how="inner"), dfs)

# Compute average CNNaffinity
cnn_cols = [f"CNNaffinity_{target}" for target in files.keys()]
merged_df["CNNaffinity_avg"] = merged_df[cnn_cols].mean(axis=1)

# Optional: sort by best (most negative) average affinity
merged_df.sort_values("CNNaffinity_avg", inplace=True)

# Save
merged_df.to_csv(output_csv, index=False)
print(f"✅ Combined and averaged table saved as: {output_csv} ({len(merged_df)} ligands)")


✅ Combined and averaged table saved as: 3_combined_all_targets.csv (25 ligands)


In [3]:
# For converting single pdbqt into multiple conforma use split_pdbqts.sh file 

In [ ]:
#for averaging the averaged csv or replicas
import pandas as pd

# Filenames of the CSVs
file1 = "1_combined_all_targets.csv"
file2 = "2_combined_all_targets.csv"
file3 = "3_combined_all_targets.csv"

# Read CSV files
csv1 = pd.read_csv(file1)
csv2 = pd.read_csv(file2)
csv3 = pd.read_csv(file3)

# Set index to Ligand for each dataframe
csv1.set_index("Ligand", inplace=True)
csv2.set_index("Ligand", inplace=True)
csv3.set_index("Ligand", inplace=True)

# Concatenate along columns with multi-level columns for distinction
merged = pd.concat([csv1, csv2, csv3], axis=1, keys=["csv1", "csv2", "csv3"])

# Average columns by their original names (second level in multi-index)
avg_df = merged.groupby(level=1, axis=1).mean()

# Reset index to bring back Ligand as a column
avg_df.reset_index(inplace=True)

# Save the averaged dataframe to a new CSV file
avg_df.to_csv("averaged_targets.csv", index=False)


/tmp/ipykernel_60864/250788143.py:22: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  avg_df = merged.groupby(level=1, axis=1).mean()
